In [1]:
"""
PicScan - CIFAR-10 CNN Training Script (v2: augmentation + deeper architecture)
================================================================================
Improves on the 87.3% baseline using:
  1. In-model data augmentation layers (auto-disabled at inference/predict time)
  2. A deeper Conv2D + BatchNorm + MaxPool + Dropout stack
  3. ModelCheckpoint + ReduceLROnPlateau + EarlyStopping
  4. Up to 100 epochs, validation_split = 0.1

Run in Google Colab or locally with TensorFlow >= 2.15.
"""

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np

# ---------------------------------------------------------------------------
# 1. Reproducibility
# ---------------------------------------------------------------------------
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

# ---------------------------------------------------------------------------
# 2. Load and preprocess CIFAR-10
# ---------------------------------------------------------------------------
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

class_names = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck",
]

# Normalize pixel values to [0, 1]. Keep as float32 for numerical stability.
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# Labels are shape (N, 1) by default -> flatten to (N,) for sparse losses/metrics.
y_train = y_train.squeeze()
y_test = y_test.squeeze()

input_shape = x_train.shape[1:]  # (32, 32, 3)
num_classes = 10

print(f"Train: {x_train.shape}, Test: {x_test.shape}")

# ---------------------------------------------------------------------------
# 3. Data augmentation as Keras layers (part of the model graph)
# ---------------------------------------------------------------------------
# These layers only apply during training (model.fit). During model.evaluate,
# model.predict, or when you load the saved model in Django and call
# model.predict(), Keras automatically sets training=False and skips them.
data_augmentation = keras.Sequential(
    [
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.08),          # gentle rotation
        layers.RandomZoom(0.1),               # zoom in/out up to 10%
        layers.RandomTranslation(0.1, 0.1),   # shift up to 10% width/height
    ],
    name="data_augmentation",
)

# ---------------------------------------------------------------------------
# 4. Model architecture
# ---------------------------------------------------------------------------
def build_model():
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)

    # ---- Block 1: 32 filters ----
    x = layers.Conv2D(32, (3, 3), padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Conv2D(32, (3, 3), padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Dropout(0.2)(x)

    # ---- Block 2: 64 filters ----
    x = layers.Conv2D(64, (3, 3), padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Conv2D(64, (3, 3), padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Dropout(0.3)(x)

    # ---- Block 3: 128 filters ----
    x = layers.Conv2D(128, (3, 3), padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Conv2D(128, (3, 3), padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Dropout(0.4)(x)

    # ---- Classifier head ----
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    return keras.Model(inputs, outputs, name="picscan_cnn_v2")


model = build_model()
model.summary()

# ---------------------------------------------------------------------------
# 5. Compile
# ---------------------------------------------------------------------------
# NOTE: label_smoothing removed here — SparseCategoricalCrossentropy does not
# support that parameter in the current stable Keras release, and including
# it would crash this line with a TypeError.
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"],
)

# ---------------------------------------------------------------------------
# 6. Callbacks
# ---------------------------------------------------------------------------
callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath="picscan_best_model.keras",
        monitor="val_loss",
        save_best_only=True,
        verbose=1,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=1,
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=12,
        restore_best_weights=True,
        verbose=1,
    ),
]

# ---------------------------------------------------------------------------
# 7. Train
# ---------------------------------------------------------------------------
BATCH_SIZE = 64
EPOCHS = 100

history = model.fit(
    x_train,
    y_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_split=0.1,
    callbacks=callbacks,
    verbose=1,
)

# ---------------------------------------------------------------------------
# 8. Evaluate on the held-out test set
# ---------------------------------------------------------------------------
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"\nFinal test accuracy: {test_acc * 100:.2f}%")
print(f"Final test loss: {test_loss:.4f}")

# ---------------------------------------------------------------------------
# 9. Save the final model for Django integration
# ---------------------------------------------------------------------------
# picscan_best_model.keras (saved by ModelCheckpoint) already holds the best
# weights by val_loss. This line additionally saves the in-memory model,
# which — thanks to restore_best_weights=True — is also the best-weights
# version at this point. Either file is safe to load in Django.
model.save("picscan_final_model.keras")

# ---------------------------------------------------------------------------
# 10. Example of how Django will later load and use this model
# ---------------------------------------------------------------------------
# from tensorflow import keras
# model = keras.models.load_model("picscan_final_model.keras")
# prediction = model.predict(preprocessed_image_batch)   # augmentation layers
#                                                          # are inactive here
# predicted_class = class_names[prediction.argmax(axis=1)[0]]

A local file was found, but it seems to be incomplete or outdated because the auto file hash does not match the original value of 6d958be074577803d12ecdefd02955f39262c83c16fe9348329d7fe0b5c001ce so we will re-download the data.
170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 2854s 17us/step
Train: (50000, 32, 32, 3), Test: (10000, 32, 32, 3)


Model: "picscan_cnn_v2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 32, 32, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ data_augmentation (Sequential)  │ (None, 32, 32, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 32, 32, 32)     │           864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 32, 32, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 32, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 32, 32, 32)     │         9,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 32, 32, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 32, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 16, 16, 64)     │        18,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 16, 16, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 16, 16, 64)     │        36,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 16, 16, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_3 (Activation)       │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 8, 8, 128)      │        73,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 8, 8, 128)      │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_4 (Activation)       │ (None, 8, 8, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 8, 8, 128)      │       147,456 │
├─────────────────────────────────┼────────────────────────┼─────────────

 Total params: 306,538 (1.17 MB)

 Trainable params: 305,386 (1.16 MB)

 Non-trainable params: 1,152 (4.50 KB)

Epoch 1/100
703/704 ━━━━━━━━━━━━━━━━━━━━ 0s 124ms/step - accuracy: 0.2872 - loss: 2.0172
Epoch 1: val_loss improved from None to 2.05940, saving model to picscan_best_model.keras
704/704 ━━━━━━━━━━━━━━━━━━━━ 94s 128ms/step - accuracy: 0.3554 - loss: 1.7884 - val_accuracy: 0.3650 - val_loss: 2.0594 - learning_rate: 0.0010
Epoch 2/100
703/704 ━━━━━━━━━━━━━━━━━━━━ 0s 134ms/step - accuracy: 0.4650 - loss: 1.4779
Epoch 2: val_loss improved from 2.05940 to 1.90784, saving model to picscan_best_model.keras
704/704 ━━━━━━━━━━━━━━━━━━━━ 97s 138ms/step - accuracy: 0.4866 - loss: 1.4261 - val_accuracy: 0.4184 - val_loss: 1.9078 - learning_rate: 0.0010
Epoch 3/100
703/704 ━━━━━━━━━━━━━━━━━━━━ 0s 134ms/step - accuracy: 0.5280 - loss: 1.3216
Epoch 3: val_loss improved from 1.90784 to 1.84805, saving model to picscan_best_model.keras
704/704 ━━━━━━━━━━━━━━━━━━━━ 97s 138ms/step - accuracy: 0.5403 - loss: 1.2929 - val_accuracy: 0.4742 - val_loss: 1.8481 - learning_rate: 0.0010
Epoch 4/100
703/704 ━━━━━

KeyboardInterrupt: 